### Question 1 :
Download the fashion-MNIST dataset and plot 1 sample image for each class as shown in the grid below. Use from keras.datasets import fashion_mnist for getting the fashion mnist dataset.

In [4]:
from keras.datasets import fashion_mnist
import numpy as np
import wandb


wandb.init(project="DL_Assignment_1", name="question_1")


(x_train, y_train), (_, _) = fashion_mnist.load_data()


class_labels = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat","Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]


images_per_class = {}

for cls in np.unique(y_train):  
    index = np.where(y_train == cls)[0][0] 
    images_per_class[class_labels[cls]] = x_train[index]  


wandb.log({"Class-wise Images": [wandb.Image(img, caption=label) for label, img in images_per_class.items()]})

wandb.finish()


### Question 2
Implement a feedforward neural network which takes images from the fashion-mnist data as input and outputs a probability distribution over the 10 classes.
Your code should be flexible such that it is easy to change the number of hidden layers and the number of neurons in each hidden layer.

In [ ]:
import math
import random
import numpy as np
from keras.datasets import fashion_mnist
import numpy as np
import matplotlib.pyplot as plt


(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()

class_labels = [
    "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"
]

def to_one_hot(labels, num_classes):
    one_hot = np.zeros((len(labels), num_classes)) 
    one_hot[np.arange(len(labels)), labels] = 1    
    return one_hot

y_train = to_one_hot(y_train, 10)
y_test = to_one_hot(y_test, 10)
x_train = x_train.reshape(x_train.shape[0], -1)
x_test = x_test.reshape(x_test.shape[0], -1)
x_train = x_train / 255.0
x_test = x_test / 255.0



def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True)) 
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


def cross_entropy_loss(y_true, y_pred):
    return -np.sum(y_true * np.log(y_pred + 1e-8)) / y_true.shape[0]


def compute_accuracy(y_true, y_pred):
    true_labels = np.argmax(y_true, axis=1)
    predicted_labels = np.argmax(y_pred, axis=1)
    return np.mean(true_labels == predicted_labels) * 100


class NeuralNetwork:
    def __init__(self, layers):
        self.layers = layers
        self.weights = []
        self.biases = []


        for i in range(len(layers) - 1):

            self.weights.append(np.random.randn(self.layers[i], self.layers[i+1]) * np.sqrt(2 / self.layers[i]))
            self.biases.append(np.zeros((1, layers[i+1])))

    def forward(self, X):
        activations = [X]
        for i in range(len(self.weights) - 1): 
            a = np.dot(activations[-1], self.weights[i]) + self.biases[i]
            h = sigmoid(a)
            activations.append(h)

       
        a = np.dot(activations[-1], self.weights[-1]) + self.biases[-1]
        h = softmax(a)
        activations.append(h)

        return activations

    def backward(self, X, y_true, activations):
        gradients_w = [None] * len(self.weights)
        gradients_b = [None] * len(self.biases)

       
        error = activations[-1] - y_true

        for i in reversed(range(len(self.weights))):
            gradients_w[i] = np.matmul(activations[i].T, error) / X.shape[0]
            gradients_b[i] = np.sum(error, axis=0, keepdims=True) / X.shape[0]

            if i > 0:
                error = np.dot(error, self.weights[i].T) * sigmoid_derivative(activations[i])

        return gradients_w, gradients_b

    def update_parameters(self, gradients_w, gradients_b, learning_rate):
        for i in range(len(self.weights)):
            self.weights[i] -= learning_rate * gradients_w[i]
            self.biases[i] -= learning_rate * gradients_b[i]

    def train(self, X_train, y_train,x_test,y_test, epochs=1000, learning_rate=0.01, offset=10):
        for epoch in range(epochs):
            activations = self.forward(X_train)
            loss = cross_entropy_loss(y_train, activations[-1])
            test_activations = self.forward(x_test)
            accuracy = compute_accuracy(y_test, test_activations[-1])

            gradients_w, gradients_b = self.backward(X_train, y_train, activations)
            self.update_parameters(gradients_w, gradients_b, learning_rate)

            if epoch % offset == 0:
                print(f"Epoch {epoch}, Loss: {loss:.4f}, Accuracy: {accuracy:.2f}%")

    def predict(self, X):
        activations = self.forward(X)
        return np.argmax(activations[-1], axis=1)


layers = [784,32,64,32,10]
nn = NeuralNetwork(layers)

In [ ]:
nn.train(x_train[:50000] , y_train[:50000],x_train[50001:],y_train[50001:], epochs=500, learning_rate=0.5,offset=100)